# Diabetes Prediction — Exploratory Data Analysis
**Dataset:** Pima Indians Diabetes Dataset (Kaggle)  
**Goal:** Understand the data before building machine learning models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load the Dataset

In [ ]:
df = pd.read_csv('../data/diabetes.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Information

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 3. Class Distribution (Diabetic vs Non-Diabetic)

In [ ]:
counts = df['Outcome'].value_counts()
labels = ['Not Diabetic (0)', 'Diabetic (1)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(labels, counts.values, color=['#4ade80', '#f87171'], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution — Count')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#4ade80', '#f87171'], startangle=90)
axes[1].set_title('Class Distribution — Percentage')

plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Missing Values (Zeros as Missing Data)
In this dataset, `0` in columns like Glucose, BloodPressure, BMI etc. is biologically impossible and represents missing data.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

zero_counts = pd.DataFrame({
    'Column': zero_cols,
    'Zero Count': [(df[col] == 0).sum() for col in zero_cols],
    'Zero %': [(df[col] == 0).mean() * 100 for col in zero_cols]
})
zero_counts['Zero %'] = zero_counts['Zero %'].round(1)
print(zero_counts.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(zero_counts['Column'], zero_counts['Zero %'], color='#60a5fa', edgecolor='white')
plt.title('Percentage of Missing Values (Zeros) per Column', fontsize=13, fontweight='bold')
plt.ylabel('Percentage (%)')
plt.xlabel('Feature')
for i, row in zero_counts.iterrows():
    plt.text(i, row['Zero %'] + 0.5, f"{row['Zero %']}%", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../models/plot_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Replace Missing Values with Median

In [ ]:
df_clean = df.copy()
for col in zero_cols:
    median = df_clean[col].replace(0, np.nan).median()
    df_clean[col] = df_clean[col].replace(0, median)

print('Missing values replaced with column medians.')
df_clean.describe().round(2)

## 6. Feature Distributions

In [ ]:
features = df_clean.columns[:-1]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df_clean[df_clean['Outcome'] == 0][col], alpha=0.6,
                 bins=20, color='#4ade80', label='Not Diabetic')
    axes[i].hist(df_clean[df_clean['Outcome'] == 1][col], alpha=0.6,
                 bins=20, color='#f87171', label='Diabetic')
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Outcome', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../models/plot_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Boxplots — Outlier Detection

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    df_clean.boxplot(column=col, by='Outcome', ax=axes[i],
                     boxprops=dict(color='#1a56db'),
                     medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col)
    axes[i].set_xlabel('Outcome (0=No, 1=Yes)')

plt.suptitle('Boxplots: Feature vs Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/plot_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key Observations

- **Class imbalance:** ~65% non-diabetic vs ~35% diabetic patients.
- **Missing data:** Insulin (48.7%) and SkinThickness (29.6%) have the most missing values encoded as zeros — replaced with median.
- **Glucose** has the strongest positive correlation with Outcome (0.49), making it the most predictive feature.
- **BMI** and **Age** also show strong separation between diabetic and non-diabetic groups.
- **Insulin** and **SkinThickness** show significant outliers.

These insights justify the need for preprocessing and guide feature importance expectations in the models.